<a href="https://colab.research.google.com/github/saumya8b-dev/awesome-website/blob/main/aimodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#libraries
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
#data set
target_heights = np.linspace(150, 170, 17)

# 2. Define the standard mathematical BMI anchors per size
size_rules = {"S": 19.5, "M": 23.5, "L": 27.5}

data = []

# 3. Create the 51-row cross-product grid loop (3 sizes * 17 heights = 51)
for size, target_bmi in size_rules.items():
    for h_cm in target_heights:
        h_cm_rounded = round(float(h_cm), 1)
        h_m = h_cm_rounded / 100.0
        w_kg = round(target_bmi * (h_m**2), 1)

        data.append(
            {"Height_cm": h_cm_rounded, "Weight_kg": w_kg, "Size": size}
        )

# 4. Construct and display the finalized DataFrame
df = pd.DataFrame(data)

X = df.drop('Size', axis = 1).values
Y = df['Size'].values
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.33,      # 20%-40% for testing
    random_state=42,     # Keeps the split identical every run
    stratify=Y           # Ensures S, M, and L are equally distributed in both sets
)
knn = KNeighborsClassifier(n_neighbors = 3)
knn.fit(X_train, Y_train)
prediction = knn.predict(X_test)
print(prediction)
print('Accuracy: ', knn.score(X_test, Y_test))

# 1. Accept interactive inputs from the terminal
user_height = float(input("Enter Height in cm (e.g., 162): "))
user_weight = float(input("Enter Weight in kg (e.g., 55): "))

# 2. Format inputs into a matching 2D structure
new_data = pd.DataFrame([[user_height, user_weight]], columns=["Height_cm", "Weight_kg"])



# 4. Generate prediction using your pre-trained KNN model
predicted_size = knn.predict(new_data)[0]

print(f"\nPredicted Clothing Size: {predicted_size}")



['L' 'L' 'M' 'S' 'M' 'S' 'S' 'L' 'S' 'M' 'M' 'M' 'L' 'S' 'M' 'L' 'S']
Accuracy:  1.0
Enter Height in cm (e.g., 162): 170
Enter Weight in kg (e.g., 55): 85

Predicted Clothing Size: L


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# 1. Set seed for reproducible messiness
np.random.seed(10)

# 2. Generate the base 51-row grid
target_heights = np.linspace(150, 170, 17)
size_rules = {"S": 19.5, "M": 23.5, "L": 27.5}
data = []

for size, target_bmi in size_rules.items():
    for h_cm in target_heights:
        h_cm_rounded = round(float(h_cm), 1)
        h_m = h_cm_rounded / 100.0

        # Calculate perfect base weight
        perfect_weight = target_bmi * (h_m**2)

        # INJECT HUMAN NOISE: Add or subtract random weight variance up to 4.5 kg
        real_human_weight = round(
            perfect_weight + np.random.uniform(-4.5, 4.5), 1
        )

        data.append(
            {"Height": h_cm_rounded, "Weight": real_human_weight, "Size": size}
        )

df_messy = pd.DataFrame(data)

# 3. Separate features and targets
X = df_messy[["Height", "Weight"]].values
Y = df_messy["Size"].values

# 4. Split into Train and Test
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.33, random_state=42, stratify=Y
)

# 5. Train KNN
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, Y_train)

# 6. Evaluate
print("--- Real-World Simulated Performance ---")
print(f"Realistic Accuracy: {knn.score(X_test, Y_test) * 100:.1f}%\n")
print(classification_report(Y_test, knn.predict(X_test)))


--- Real-World Simulated Performance ---
Realistic Accuracy: 94.1%

              precision    recall  f1-score   support

           L       1.00      1.00      1.00         5
           M       0.86      1.00      0.92         6
           S       1.00      0.83      0.91         6

    accuracy                           0.94        17
   macro avg       0.95      0.94      0.94        17
weighted avg       0.95      0.94      0.94        17



In [6]:
#step 1: import
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
#step 2 : data


# 1. The official raw GitHub link to the JHU global confirmed cases CSV
DATA_URL = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"

# 2. Pandas downloads and reads the file into memory instantly
print("Fetching live data from GitHub...")
df = pd.read_csv(DATA_URL)

# 3. Filter for India (extracting the entire national timeline)
india_df = df[df["Country/Region"] == "India"]

# 4. Strip out metadata columns (leaving only date columns)
columns_to_drop = ["Province/State", "Country/Region", "Lat", "Long"]
india_timeline = india_df.drop(columns=columns_to_drop).squeeze()

# 5. Restructure into rows instead of columns
covid_cases = india_timeline.reset_index()
covid_cases.columns = ["Date", "Cases"]

# 6. Generate the sequential day counter required for Scikit-Learn
covid_cases["Day_Number"] = range(1, len(covid_cases) + 1)

# Preview the extraction output
print("\nExtraction complete! Here are the first few rows:")
print(covid_cases.head(15))
X = covid_cases[["Day_Number"]].values
y = covid_cases["Cases"].values
#3.split the model
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
#4: train model
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
model = LinearRegression()
model.fit(X_train_poly, y_train)
#5: evaluate
y_pred = model.predict(X_test_poly)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"--- Model Evaluation ---")
print(f"Average Error on Test Days: {int(mae):,} cases")
print(f"R² Score Accuracy: {r2:.4f}\n")


Fetching live data from GitHub...

Extraction complete! Here are the first few rows:
       Date  Cases  Day_Number
0   1/22/20      0           1
1   1/23/20      0           2
2   1/24/20      0           3
3   1/25/20      0           4
4   1/26/20      0           5
5   1/27/20      0           6
6   1/28/20      0           7
7   1/29/20      0           8
8   1/30/20      1           9
9   1/31/20      1          10
10   2/1/20      1          11
11   2/2/20      2          12
12   2/3/20      3          13
13   2/4/20      3          14
14   2/5/20      3          15
--- Model Evaluation ---
Average Error on Test Days: 13,658,799 cases
R² Score Accuracy: -6100.9448



In [26]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

DATA_URL = 'https://raw.githubusercontent.com/YBI-Foundation/Dataset/refs/heads/main/House%20Prices.csv'

print("Fetching house price dataset...")
# Load the raw file safely
df = pd.read_csv(DATA_URL, sep=',', on_bad_lines='skip')

# Remove rows containing empty or invalid cells
df_clean = df.dropna()

# 🔥 FIX: Grab data by position instead of names!
# .iloc[:, [0]] grabs the VERY FIRST column (Square Footage / Area)
# .iloc[:, 1] grabs the SECOND column (Price)
X = df_clean[['Sqft_living', 'Bedrooms', 'Bathrooms']]
Y = df_clean[['Price']]

# Split your data
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    random_state=42, test_size=0.2
)

# Train the model
model = LinearRegression()
model.fit(X_train, Y_train)

# Make predictions and calculate performance metrics
Y_pred = model.predict(X_test)
mse = mean_squared_error(Y_test, Y_pred)
mae = mean_absolute_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)

print("\n📊 --- Model Performance Report ---")
print(f"Mean Squared Error (MSE)      : {mse:,.2f}")
print(f"Mean Absolute Error (MAE)     : ${mae:,.2f}")
print(f"R-squared (R²) Score Accuracy : {r2:.4f}")


Fetching house price dataset...

📊 --- Model Performance Report ---
Mean Squared Error (MSE)      : 64,148,882,088.94
Mean Absolute Error (MAE)     : $170,727.48
R-squared (R²) Score Accuracy : 0.4639
